### Imports
Standard data handling (`numpy`, `pandas`, `seaborn`), plus the sklearn/xgboost pieces used later: `train_test_split` for the train/validation split, `OrdinalEncoder` + `ColumnTransformer` for encoding categorical columns, `XGBRegressor` as the baseline model, and `r2_score`/`root_mean_squared_error` for evaluation (R2 is the competition's actual metric, RMSE is a more intuitive secondary check).

In [25]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

### Load raw data
Read the competition's train and test CSVs.

In [26]:
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")

df_train

,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,a,d,u,j,o,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,e,d,y,l,o,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,c,d,x,j,x,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,az,t,n,f,d,x,l,e,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,az,v,n,f,d,h,d,n,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4204,8405,107.39,ak,s,as,c,d,aa,d,q,...,1,0,0,0,0,0,0,0,0,0
4205,8406,108.77,j,o,t,d,d,aa,h,h,...,0,1,0,0,0,0,0,0,0,0
4206,8412,109.22,ak,v,r,a,d,aa,g,e,...,0,0,1,0,0,0,0,0,0,0
4207,8415,87.48,al,r,e,f,d,aa,l,u,...,0,0,0,0,0,0,0,0,0,0


Preview the test set — note it has no `y` column, since predicting `y` is the task.

In [27]:
df_test

,ID,X0,X1,X2,X3,X4,X5,X6,X8,X10,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,1,az,v,n,f,d,t,a,w,0,...,0,0,0,1,0,0,0,0,0,0
1,2,t,b,ai,a,d,b,g,y,0,...,0,0,1,0,0,0,0,0,0,0
2,3,az,v,as,f,d,a,j,j,0,...,0,0,0,1,0,0,0,0,0,0
3,4,az,l,n,f,d,z,l,n,0,...,0,0,0,1,0,0,0,0,0,0
4,5,w,s,as,c,d,y,i,m,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4204,8410,aj,h,as,f,d,aa,j,e,0,...,0,0,0,0,0,0,0,0,0,0
4205,8411,t,aa,ai,d,d,aa,j,y,0,...,0,1,0,0,0,0,0,0,0,0
4206,8413,y,v,as,f,d,aa,d,w,0,...,0,0,0,0,0,0,0,0,0,0
4207,8414,ak,v,as,a,d,aa,c,q,0,...,0,0,1,0,0,0,0,0,0,0


### Split target from features
`ID` is just a row identifier with no predictive value, so it's dropped from the feature set `X`. `test_ID` is kept separately since it's needed to build the Kaggle submission file later.

In [28]:
y = df_train['y']
X = df_train.drop(['ID', 'y'], axis = 1)

test_ID = df_test['ID']
X_test = df_test.drop(['ID'], axis = 1)

### Train/validation split
Holds out 20% of the training data as `X_val`/`y_val`, standing in for genuinely unseen data, so we can honestly measure generalization before ever touching the real `X_test`.

In [29]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)

### Exploring numeric columns
All 368 non-categorical columns turn out to be binary (0/1). This split is for exploration only — the actual `cat_cols`/`num_cols` used in the pipeline below are recomputed from `X_train` directly, not from this cell.

In [30]:
num_features = X.select_dtypes(exclude = 'str')
num_features

,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,0,0,1,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4204,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4205,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4206,0,0,1,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4207,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Exploring categorical columns
8 text columns (`X0`-`X8`), with cardinality ranging from 4 (`X4`) up to 47 (`X0`). `X0`, `X2`, and `X5` also contain category values in `test.csv` that never appear in `train.csv` — this is what drives the encoding choice below (ordinal encoding with an explicit fallback for unseen values, instead of one-hot encoding).

In [31]:
cat_features = X.select_dtypes(include = 'str')
cat_features

,X0,X1,X2,X3,X4,X5,X6,X8
0,k,v,at,a,d,u,j,o
1,k,t,av,e,d,y,l,o
2,az,w,n,c,d,x,j,x
3,az,t,n,f,d,x,l,e
4,az,v,n,f,d,h,d,n
...,...,...,...,...,...,...,...,...
4204,ak,s,as,c,d,aa,d,q
4205,j,o,t,d,d,aa,h,h
4206,ak,v,r,a,d,aa,g,e
4207,al,r,e,f,d,aa,l,u


### Build the preprocessing pipeline
`cat_cols`/`num_cols` are computed from `X_train` (not `X`), so nothing about `X_val`'s categories leaks into what the encoder learns.

Ordinal encoding (not one-hot) is used because: cardinality up to 47 would make one-hot very wide, and tree models don't assume any real order in the integer codes, so there's no downside. `handle_unknown='use_encoded_value', unknown_value=-1` safely handles categories in `X_val`/`X_test` that were never seen while fitting on `X_train`, without needing to peek at val/test to build the mapping.

In [32]:
cat_cols = X_train.select_dtypes(include='str').columns.tolist()
num_cols = X_train.select_dtypes(exclude='str').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

### Fit and apply the preprocessing
`set_output(transform='pandas')` makes `.transform()` return a DataFrame with real column names instead of a bare numpy array. The preprocessor is fit on `X_train` only; `X_val` and `X_test` are only ever *transformed* with the mapping already learned from train, never used to fit it.

In [33]:
preprocessor.set_output(transform='pandas')
preprocessor.fit(X_train)

X_train_encoded = preprocessor.transform(X_train)
X_val_encoded   = preprocessor.transform(X_val)
X_test_encoded  = preprocessor.transform(X_test)

### Dummy Baseline

In [34]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

dummy_model = DummyRegressor(strategy='mean')
dummy_scores = cross_val_score(dummy_model, X_train_encoded, y_train, cv=kfold, scoring='r2')

print(f"Dummy R2 per fold: {dummy_scores}")
print(f"Dummy R2 mean:     {dummy_scores.mean():.4f}")

Dummy R2 per fold: [-0.00254166 -0.0013445  -0.00315743 -0.00114044 -0.00906566]
Dummy R2 mean:     -0.0034


### Baseline model
Default (untuned) `XGBRegressor` hyperparameters, so later feature engineering and tuning have an honest number to compare against. No feature scaling — unnecessary for tree models, which split one column at a time regardless of magnitude.

In [35]:
baseline_model = XGBRegressor(random_state=42)
baseline_model.fit(X_train_encoded, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### Evaluate on validation set
Predicts on `X_val`, which was never used to fit the encoder or the model. R2 is the competition's actual scoring metric; RMSE is a more intuitive "average miss" in `y`'s own units, kept as a secondary sanity check.

In [36]:
y_val_pred = baseline_model.predict(X_val_encoded)

r2 = r2_score(y_val, y_val_pred)
rmse = root_mean_squared_error(y_val, y_val_pred)

print(f"Validation R2:   {r2:.4f}")
print(f"Validation RMSE: {rmse:.4f}")

Validation R2:   0.4493
Validation RMSE: 9.2583


## Findings so far / what should inform next steps

**Baseline result**
- Default (untuned) `XGBRegressor` → Validation R2 = 0.4493, RMSE = 9.2583.
- Competition's actual metric is R2. Public leaderboard scores for this competition typically top out around 0.55-0.58, so 0.4493 is a believable, unremarkable starting point — room to improve via feature work and tuning, not a sign of a broken pipeline.

**Feature structure**
- 8 categorical (text) columns: `X0, X1, X2, X3, X4, X5, X6, X8`. Cardinality ranges from 4 (`X4`) to 47-49 (`X0`).
- 368 numeric columns, all binary (0/1) — no scaling needed for tree models regardless.
- `X0`, `X2`, `X5` have categories in `test.csv` that never appear in `train.csv` (6, 14, and 4 affected test rows respectively, out of 4209) — this is why ordinal encoding with `unknown_value=-1` was used instead of one-hot, and why the encoder is fit on `X_train` only (not `X`, not `X_test`) to avoid leaking validation/test category vocabulary into training.

**Known outlier**
- One training row (`ID=1770`) has `y=265.32`; every other row falls between 72 and 170. Because RMSE/R2 are both built from squared error, this single point can disproportionately influence the model's fit.
- Confirmed this row currently lands in `X_train` under `random_state=42`, not in `X_val` — so it isn't distorting the validation score directly, but may still be distorting what the model learns.
- **Next experiment:** drop this row from training only (leave val/test untouched) and re-fit the baseline to see whether R2 improves.

**Candidates for feature engineering / dimensionality reduction (not yet done)**
- Check the 368 binary columns for constant (zero-variance) or exact-duplicate columns — common in this specific dataset — and drop them; trees ignore uninformative columns anyway but removing them reduces noise/training time.
- Consider PCA or similar dimensionality reduction on the binary block *only if* redundancy/duplication turns out to be substantial, or if overfitting becomes visible (train score much higher than validation score) — not a given win for tree models, so should be tested against the baseline rather than applied by default.
- Any change (row removal, dropped columns, PCA, tuning) should be compared against the R2 = 0.4493 baseline above before being kept.


## Experiment: drop constant & duplicate columns

Computed from `X_train` only (not the full `train_df`), matching the fit-on-train-only discipline used for the encoder — a column being constant/duplicate across the full dataset doesn't guarantee it's still constant/duplicate within just the train split.

In [37]:
constant_cols = [c for c in X_train.columns if X_train[c].nunique() == 1]

non_constant_cols = [c for c in X_train.columns if c not in constant_cols]
dup_groups = X_train[non_constant_cols].T.groupby(
    X_train[non_constant_cols].T.apply(tuple, axis=1)
).groups
duplicate_groups = [list(v) for v in dup_groups.values() if len(v) > 1]
duplicate_cols_to_drop = [c for group in duplicate_groups for c in group[1:]]

cols_to_drop = constant_cols + duplicate_cols_to_drop

print(f"Constant columns:          {len(constant_cols)}")
print(f"Duplicate columns to drop: {len(duplicate_cols_to_drop)}")
print(f"Total columns to drop:     {len(cols_to_drop)}")

Constant columns:          13
Duplicate columns to drop: 48
Total columns to drop:     61


### Drop the identified columns from the raw features

Dropped from `X_train`/`X_val`/`X_test` *before* encoding — not from `X_train_encoded` etc. afterward — since the encoded frames have `num__`/`cat__` prefixes on their column names (added by `ColumnTransformer` + `set_output(transform='pandas')`) that wouldn't match these raw `X11`-style names.

In [38]:
X_train_clean = X_train.drop(columns=cols_to_drop)
X_val_clean   = X_val.drop(columns=cols_to_drop)
X_test_clean  = X_test.drop(columns=cols_to_drop)

X_train_clean.shape

(3367, 315)

### Rebuild and refit the preprocessing pipeline on the cleaned columns

`cat_cols`/`num_cols` are recomputed since `num_cols` is now smaller. `preprocessor_clean` is a fresh `ColumnTransformer` (not a mutation of the original `preprocessor`) so the original baseline's encoder and encoded frames stay untouched for comparison.

In [39]:
cat_cols_clean = X_train_clean.select_dtypes(include='str').columns.tolist()
num_cols_clean = X_train_clean.select_dtypes(exclude='str').columns.tolist()

preprocessor_clean = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols_clean),
        ('num', 'passthrough', num_cols_clean),
    ]
)
preprocessor_clean.set_output(transform='pandas')
preprocessor_clean.fit(X_train_clean)

X_train_clean_encoded = preprocessor_clean.transform(X_train_clean)
X_val_clean_encoded   = preprocessor_clean.transform(X_val_clean)
X_test_clean_encoded  = preprocessor_clean.transform(X_test_clean)

### Refit and compare against the baseline

Same `XGBRegressor` settings as the original baseline, fit on the cleaned feature set, so the R2 difference reflects only the column cleanup — not a change in model configuration.

In [40]:
cleaned_model = XGBRegressor(random_state=42)
cleaned_model.fit(X_train_clean_encoded, y_train)

y_val_pred_clean = cleaned_model.predict(X_val_clean_encoded)

r2_clean = r2_score(y_val, y_val_pred_clean)
rmse_clean = root_mean_squared_error(y_val, y_val_pred_clean)

print(f"Validation R2:   {r2_clean:.4f}  (baseline was {r2:.4f})")
print(f"Validation RMSE: {rmse_clean:.4f}  (baseline was {rmse:.4f})")

Validation R2:   0.4493  (baseline was 0.4493)
Validation RMSE: 9.2583  (baseline was 9.2583)
